# 07 — Run the 4-config evaluation matrix on Kaggle

Drives `scripts/run_eval.py` over configs **A / B / C / D** on the 54-question test set, then builds the blinded human-eval form. Outputs land under `experiments/results/`:

```
experiments/results/
  A_base_no_rag/predictions.jsonl   metrics.json
  B_base_with_rag/predictions.jsonl metrics.json
  C_finetuned_no_rag/predictions.jsonl metrics.json
  D_finetuned_with_rag/predictions.jsonl metrics.json
  summary.json                          # all 4 configs in one file
  human_eval/form.csv  key.csv          # blinded rating sheet
```

## Why a separate notebook (not the demo)

The demo notebook avoids loading the model into the kernel — it forks a Streamlit subprocess that owns the GPU. Eval is the opposite: each config **must** load directly in this kernel, run inference on 54 questions, then unload before the next one. `run_eval.py` calls `pipe.unload()` between configs so VRAM stays bounded to one Qwen2.5-7B at a time (≈6 GB on a T4).

## Setup before running

1. **Settings → Accelerator → GPU P100** (or **T4 ×2**, recommended for headroom).
2. **Settings → Internet → On**.
3. **Add-ons → Secrets → `HF_TOKEN`** attached.
4. **Run all cells.** Total wall-clock: ≈60–90 min for all four configs (≈50 questions × ~15–40 s/answer + BERTScore).

## Tips

- Smoke-test first with `--limit 5` (cell 6) before committing to the full run.
- BERTScore downloads `xlm-roberta-large` (≈2 GB) the first time it runs. Pass `--skip-bertscore` if you only need BLEU + ROUGE-L + Recall@5.
- If you hit OOM mid-run, the partial `predictions.jsonl` files for completed configs are still on disk — re-run with `--configs <remaining>` to continue.

In [ ]:
# 1. Clone (or update) the repo at /kaggle/working/LawMate.
import os, subprocess

REPO_URL = "https://github.com/tamir39/rag-llm-vietnam-law-advisor.git"
REPO_DIR = "/kaggle/working/LawMate"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "develop", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", "develop"])

os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "log", "-1", "--oneline"]).decode().strip())

In [ ]:
# 2. Install the LLM/RAG stack + metric libraries + hf_transfer.
#    No streamlit / cloudflared needed here — this kernel is the worker, not a server.
%pip install -q -U \
  "peft>=0.12" "trl>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33" \
  "sentence-transformers>=3.0" "faiss-cpu>=1.8" \
  "sacrebleu>=2.4" "rouge-score>=0.1.2" "bert-score>=0.3.13" \
  "hf_transfer>=0.1.8"

In [ ]:
# 3. HF login — needed for the LoRA adapter (configs C/D) and Qwen base.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF login OK")

In [ ]:
# 4. Build the FAISS index if it isn't there yet (~30s on Kaggle CPU).
#    Configs B and D need this; A and C don't, but it's cheap to always have.
import os, subprocess

if not os.path.isfile("experiments/index/kb.faiss"):
    subprocess.check_call(["python", "scripts/build_index.py"])
else:
    print("FAISS index already present — skipping build")

In [ ]:
# 5. Pre-fetch model weights with hf_transfer (3-5x faster than transformers' default).
#    SKIP this cell if you've attached Qwen2.5-7B-Instruct as a Kaggle Model.
import os, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download

REPOS = [
    "Qwen/Qwen2.5-7B-Instruct",             # base LLM (~15 GB)
    "Tamir39/qwen2_5-7b-vietnam-tax-lora",  # fine-tuned adapter (~160 MB)
    "intfloat/multilingual-e5-base",        # RAG embedder (~1 GB)
]

t0 = time.time()
for repo in REPOS:
    print(f"\n[{repo}] downloading...")
    snapshot_download(repo, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.md"])
print(f"\nAll files cached on disk in {time.time()-t0:.0f}s")

## Smoke test (optional, recommended first)

Run config **D** on 5 questions only. If this works end-to-end (load → generate → metrics dumped), the full run will work too. Saves you discovering a bug 45 minutes in.

Comment out / skip the cell below for a full run.

In [ ]:
import subprocess
subprocess.check_call([
    "python", "scripts/run_eval.py",
    "--configs", "D",
    "--limit", "5",
    "--skip-bertscore",
])

## Full evaluation — all 4 configs

≈1–1.5 hours on T4 ×2. Predictions and metrics are written incrementally per config, so a mid-run failure on config X still leaves usable output for the configs that finished before it.

Pass `--skip-bertscore` to drop ~15 minutes (BERTScore needs `xlm-roberta-large` and is the heaviest metric).

In [ ]:
import subprocess
subprocess.check_call([
    "python", "scripts/run_eval.py",
    # "--skip-bertscore",  # uncomment to save ~15 min
])

In [ ]:
# 8. Display the summary in a tidy table.
import json, pandas as pd
from pathlib import Path

summary = json.loads(Path("experiments/results/summary.json").read_text(encoding="utf-8"))
df = pd.DataFrame(summary).T
df.index.name = "config"
cols_pref = ["bleu", "rougeL", "bertscore_f1", "recall_at_5", "mrr_at_10"]
cols = [c for c in cols_pref if c in df.columns] + [c for c in df.columns if c not in cols_pref]
df = df[cols].round(4)
df

## Human-eval form (blinded)

Builds `experiments/results/human_eval/form.csv` with 50 sampled questions, each row containing the gold answer + four predictions in randomized order. The rater fills `rating_1..rating_4` (1–5). The companion `key.csv` maps slot → real config so we can de-anonymize after rating.

Download both CSVs from Kaggle's right-side **Output** panel and rate them in Excel / Google Sheets (≈1 hour for 50 questions).

In [ ]:
import subprocess
subprocess.check_call([
    "python", "scripts/build_human_eval.py",
    "--n", "50",
    "--seed", "42",
])

In [ ]:
# 10. Bundle everything into one tarball at /kaggle/working/lawmate_eval.tar.gz so you can
#     download it from the Output panel in a single click.
import shutil, os

os.makedirs("/kaggle/working", exist_ok=True)
out_path = shutil.make_archive(
    base_name="/kaggle/working/lawmate_eval",
    format="gztar",
    root_dir="experiments",
    base_dir="results",
)
size_mb = os.path.getsize(out_path) / 1024**2
print(f"Wrote {out_path} ({size_mb:.1f} MB)")

## After the run

1. Download `/kaggle/working/lawmate_eval.tar.gz` from the **Output** panel.
2. Extract it locally over `experiments/results/` and `git add -f experiments/results/summary.json experiments/results/*/metrics.json`. (The `predictions.jsonl` files can stay out of git — they're large; the human-eval CSVs are worth committing once filled.)
3. Open `docs/report/POST_TRAINING.md` and follow steps **2 → 6** to fold the numbers into the report.